![image.png](https://i.imgur.com/4fN73lZ.png)

# Directed Exploration in Deep RL: Random Network Distillation (RND)

SAC's entropy bonus (Day 6) is **undirected** exploration — it adds noise everywhere.
On a **sparse-reward** task that isn't enough: you can wander randomly forever and
never stumble on the goal. This lab adds **directed** exploration — an **intrinsic
reward** that pulls the agent toward *novel* states — via **Random Network
Distillation (RND)** (Burda et al., 2019):

- Fix a **randomly-initialized target** network $f^*$ (never trained).
- Train a **predictor** $f_\theta$ to match $f^*$ on the states we actually visit.
- The **intrinsic reward** is the prediction error
  $$r^i_t = \big\lVert f_\theta(s_t) - f^*(s_t) \big\rVert^2.$$
  A novel state has high error (the predictor hasn't learned it there yet) ⇒ high
  bonus, so the agent is drawn to explore it.

Task: **MountainCar-v0** — reward is $-1$ every step until the car reaches the flag,
which needs ~100 steps of momentum-building. Plain ε-greedy DQN almost never gets
there, so it never sees a learning signal. We run **plain DQN vs DQN+RND** head to
head. (RND's headline result is Montezuma's Revenge, but that's far too heavy for a
CPU lab — MountainCar is the standard lightweight stand-in.)

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control]" imageio matplotlib torch

In [ ]:
import collections
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device("cpu")   # tiny nets + a cheap env step -> CPU is fine
print("device:", device)

## The environment: MountainCar-v0

An underpowered car sits between two hills and must reach the flag on the right. Its
engine is too weak to drive straight up — it has to **rock back and forth to build
momentum**. Full details
[here](https://gymnasium.farama.org/environments/classic_control/mountain_car/).

- **Observation** (2 numbers): `position` $\in [-1.2, 0.6]$ and `velocity` $\in [-0.07, 0.07]$.
- **Actions** (3): push left, do nothing, push right.
- **Reward**: $-1$ on *every* step until the car reaches the flag at `position` $\geq 0.5$
  (episodes are capped at 200 steps). So the return is just $-(\text{steps taken})$, and
  there is **no learning signal at all until the agent first reaches the flag** — which
  is exactly why exploration is the whole challenge here.

![MountainCar](https://gymnasium.farama.org/_images/mountain_car.gif)

## Input normalization

MountainCar's two observations live on **very different scales**: position spans
$[-1.2, 0.6]$ while velocity spans only $[-0.07, 0.07]$. Fed raw, the velocity is
almost invisible to the networks. This matters *especially* for RND: the novelty
signal is a distance in the target network's input space, so if one axis dominates
numerically, the notion of "novel" is distorted. We rescale both observations to
roughly $[-1, 1]$ before they touch any network.

In [ ]:
OBS_LOW  = np.array([-1.2, -0.07], dtype=np.float32)
OBS_HIGH = np.array([ 0.6,  0.07], dtype=np.float32)
OBS_MID  = (OBS_LOW + OBS_HIGH) / 2
OBS_HALF = (OBS_HIGH - OBS_LOW) / 2

def norm(s):
    """Rescale a raw observation to roughly [-1, 1] per dimension."""
    return (np.asarray(s, dtype=np.float32) - OBS_MID) / OBS_HALF

## Replay buffer & DQN

Standard off-policy DQN machinery from Day 2 — a replay buffer, a Q-network, and a
target network. Nothing RND-specific yet.

In [ ]:
Experience = collections.namedtuple("Experience", ["s", "a", "r", "s2", "done"])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)
    def __len__(self):
        return len(self.buffer)
    def add(self, e):
        self.buffer.append(e)
    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        S, A, R, S2, D = zip(*[self.buffer[i] for i in idx])
        t = lambda x: torch.tensor(np.array(x, dtype=np.float32), device=device)
        return t(S), torch.tensor(A, device=device), t(R), t(S2), t(D)


class QNet(nn.Module):
    """Q(s, a): state -> one value per action."""
    def __init__(self, obs_dim=2, n_actions=3, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(), nn.Linear(h, n_actions))
    def forward(self, x):
        return self.net(x)

## The RND novelty module

Two identical small MLPs. The **target** $f^*$ is randomly initialized and **frozen**
(no gradients, ever). The **predictor** $f_\theta$ is trained to match it. On a state
the network has seen often the predictor matches the target well (low error); on a
novel state it does not (high error) — and that error *is* the novelty signal.
(In code we *average* the squared error over the feature dimension; that is
$\lVert f_\theta - f^*\rVert^2$ up to a constant factor, which the reward
normalization below absorbs.)

`RunningStd` tracks the running standard deviation of the novelty so we can
**normalize** it: the raw novelty scale drifts as the predictor learns, so a fixed
bonus weight would otherwise be meaningless.

In [ ]:
class RNDNet(nn.Module):
    def __init__(self, obs_dim=2, out_dim=64, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, h), nn.ReLU(), nn.Linear(h, out_dim))
    def forward(self, x):
        return self.net(x)


class RunningStd:
    """Welford running mean/variance, used to normalize the intrinsic reward."""
    def __init__(self):
        self.mean, self.var, self.count = 0.0, 1.0, 1e-4
    def update(self, x):
        b_mean = float(np.mean(x)); self.count += 1
        self.mean += (b_mean - self.mean) / self.count
        self.var  += (np.mean((x - self.mean) ** 2) - self.var) / self.count
    @property
    def std(self):
        return np.sqrt(self.var) + 1e-8


def rnd_novelty(predictor, target, states):
    """RND novelty on a batch of states. Returns (per-sample novelty, predictor loss)."""
    with torch.no_grad():
        f_target = target(states)          # frozen target features
    f_pred = predictor(states)             # predictor features (trainable)
    # novelty = squared predictor-vs-target error, averaged over the feature dims
    per_sample_err = ((f_pred - f_target) ** 2).mean(dim=1)
    novelty = per_sample_err.detach()      # used as a reward -> detach (no grad)
    predictor_loss = per_sample_err.mean() # trains f_theta toward f_star
    return novelty, predictor_loss

## The agent

`Agent` wraps the DQN and (optionally) the RND novelty module. Two small methods keep
the training loop below trivial to read:

- `greedy_action(s)` — the exploiting action (the loop adds ε-greedy randomness).
- `update()` — one gradient step: sample a batch, (if RND) add the normalized novelty
  bonus to the reward and train the predictor, then do the usual DQN update.

Plain DQN is just `Agent(cfg, use_rnd=False)` — identical code with the bonus turned
off.

In [ ]:
class Agent:
    def __init__(self, cfg, use_rnd):
        self.cfg, self.use_rnd = cfg, use_rnd
        self.q, self.q_target = QNet().to(device), QNet().to(device)
        self.q_target.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg["lr"])
        self.buffer = ReplayBuffer(cfg["buffer_size"])
        self.learn_steps = 0
        if use_rnd:
            self.rnd_target = RNDNet().to(device)
            for p in self.rnd_target.parameters():
                p.requires_grad_(False)                 # freeze the target
            self.rnd_predictor = RNDNet().to(device)
            self.rnd_opt = optim.Adam(self.rnd_predictor.parameters(), lr=cfg["lr"])
            self.ri_stats = RunningStd()

    def greedy_action(self, state):
        with torch.no_grad():
            return int(self.q(torch.tensor(norm(state)).unsqueeze(0)).argmax(1).item())

    def update(self):
        if len(self.buffer) < self.cfg["learn_start"]:
            return
        S, A, R, S2, D = self.buffer.sample(self.cfg["batch_size"])

        if self.use_rnd:
            novelty, pred_loss = rnd_novelty(self.rnd_predictor, self.rnd_target, S2)
            self.ri_stats.update(novelty.numpy())
            bonus = novelty / self.ri_stats.std                 # normalized intrinsic reward
            # add the novelty bonus (weighted by beta) to the extrinsic reward -- the crux of RND
            R = R + self.cfg["beta"] * bonus
            self.rnd_opt.zero_grad(); pred_loss.backward(); self.rnd_opt.step()

        with torch.no_grad():
            y = R + self.cfg["gamma"] * (1 - D) * self.q_target(S2).max(1).values
        q_sa = self.q(S).gather(1, A.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(q_sa, y)
        self.opt.zero_grad(); loss.backward(); self.opt.step()

        self.learn_steps += 1
        if self.learn_steps % self.cfg["target_sync"] == 0:
            self.q_target.load_state_dict(self.q.state_dict())

## Configuration — experiment here

Every knob for the run lives here. The defaults reliably show the effect on CPU in a
few minutes. Good things to try: set `beta = 0` to *disable* RND (recovers plain DQN);
raise `episodes` for a stronger RND policy; lower `beta` to see curiosity fade.

In [ ]:
config = {
    # --- training budget ---
    "episodes":    400,     # training episodes (RND typically first reaches the flag ~ep 200)
    # --- exploration ---
    "beta":        1.0,     # intrinsic-reward weight (0 -> plain DQN; higher -> more curiosity)
    "eps_start":   1.0,     # initial epsilon for epsilon-greedy
    "eps_min":     0.02,    # minimum epsilon
    "eps_decay":   0.995,   # per-episode multiplicative decay
    # --- DQN ---
    "gamma":       0.99,    # discount factor
    "lr":          1e-3,    # learning rate (shared by the Q-net and the RND predictor)
    "batch_size":  128,
    "buffer_size": 50_000,
    "learn_start": 1_000,   # collect this many steps of data before learning starts
    "target_sync": 500,     # gradient steps between target-network syncs
}

## Training loop

Thin and readable: act (ε-greedy), step, store, learn. We also record the **furthest
position reached** each episode — the clearest single measure of *how far the agent
explored* toward the flag at position $0.5$.

In [ ]:
def train(agent, cfg, seed=0, verbose=True):
    env = gym.make("MountainCar-v0")
    eps = cfg["eps_start"]
    returns, successes, max_positions, visited = [], [], [], []

    for ep in range(cfg["episodes"]):
        s, _ = env.reset(seed=seed * 10_000 + ep)
        done, ep_return, reached_flag, furthest = False, 0.0, False, -np.inf
        while not done:
            if np.random.random() < eps:
                a = env.action_space.sample()          # explore (epsilon-greedy)
            else:
                a = agent.greedy_action(s)             # exploit
            s2, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            reached_flag = reached_flag or terminated
            agent.buffer.add(Experience(norm(s), a, r, norm(s2), float(terminated)))
            visited.append(s2.copy())
            furthest = max(furthest, s2[0])
            s = s2; ep_return += r
            agent.update()

        eps = max(cfg["eps_min"], eps * cfg["eps_decay"])
        returns.append(ep_return); successes.append(float(reached_flag)); max_positions.append(furthest)
        if verbose and (ep + 1) % 50 == 0:
            print(f"  ep {ep+1:4d} | return(avg50)={np.mean(returns[-50:]):7.1f} "
                  f"| reached-flag(avg50)={np.mean(successes[-50:]):.2f} "
                  f"| furthest(avg50)={np.mean(max_positions[-50:]):+.2f} | eps={eps:.2f}")
    env.close()
    return dict(returns=np.array(returns), successes=np.array(successes),
                max_positions=np.array(max_positions), visited=np.array(visited))

## Run: plain DQN vs DQN + RND

Same config, same seed — the only difference is `use_rnd`.

In [ ]:
def run(use_rnd, seed=0):
    np.random.seed(seed); torch.manual_seed(seed)   # seed BEFORE building the nets
    agent = Agent(config, use_rnd)
    return agent, train(agent, config, seed=seed)

print("Plain DQN (epsilon-greedy only):")
plain_agent, plain = run(use_rnd=False)

print("\nDQN + RND (directed exploration):")
rnd_agent, rnd = run(use_rnd=True)

## Learning curves

Three views of the same story. Plain DQN flatlines: it never reaches the flag, so its
return sits at $-200$, its success rate at $0$, and the furthest position it reaches
never approaches $0.5$. RND discovers the flag and all three climb.

In [ ]:
def moving_avg(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
panels = [("returns", "return", None),
          ("successes", "reached flag", None),
          ("max_positions", "furthest position", 0.5)]
for axis, (key, ylabel, flag) in zip(ax, panels):
    axis.plot(moving_avg(plain[key]), label="plain DQN")
    axis.plot(moving_avg(rnd[key]),   label="DQN + RND")
    if flag is not None:
        axis.axhline(flag, color="red", ls="--", lw=1, label="flag (0.5)")
    axis.set_xlabel("episode"); axis.set_ylabel(f"{ylabel} (20-ep avg)")
    axis.set_title(ylabel); axis.legend(); axis.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Where each agent explored

State-visitation over $(\text{position}, \text{velocity})$, on a **log color scale** so
that rarely-visited cells are still visible (both agents spend most steps near the
start, which would otherwise wash everything else out). Plain DQN's mass never crosses
the flag line; RND's spreads rightward and reaches it.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharex=True, sharey=True)
for axis, vis, title in [(ax[0], plain["visited"], "plain DQN"),
                         (ax[1], rnd["visited"], "DQN + RND")]:
    axis.hist2d(vis[:, 0], vis[:, 1], bins=60,
                range=[[-1.2, 0.6], [-0.07, 0.07]], cmap="viridis",
                norm=LogNorm(), cmin=1)                 # log scale; hide empty bins
    axis.axvline(0.5, color="red", ls="--", lw=1.5, label="flag")
    axis.set_xlabel("position"); axis.set_title(f"visitation: {title}")
    axis.legend(loc="upper left")
ax[0].set_ylabel("velocity")
plt.tight_layout(); plt.show()

## Watch both agents

The clearest comparison of all: the same greedy rollout for each. Plain DQN rocks back
and forth and never crests; RND builds momentum and reaches the flag.

In [ ]:
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless pygame rendering
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

def render_gif(q_net, path, seed=0, steps=200):
    env = gym.make("MountainCar-v0", render_mode="rgb_array")
    s = env.reset(seed=seed)[0]
    frames, reached = [], False
    for _ in range(steps):
        frames.append(env.render())
        with torch.no_grad():
            a = int(q_net(torch.tensor(norm(s)).unsqueeze(0)).argmax(1).item())
        s, r, terminated, truncated, _ = env.step(a)
        reached = reached or terminated
        if terminated or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=30, loop=0)
    return path, reached

for label, agent in [("Plain DQN", plain_agent), ("DQN + RND", rnd_agent)]:
    path, reached = render_gif(agent.q, f"video/{label.split()[0].lower()}_mountaincar.gif")
    print(f"{label}: {'REACHED the flag' if reached else 'did NOT reach the flag'}")
    display(Image(filename=path))

## Takeaways

- On a **sparse-reward** task, undirected exploration (ε-greedy here; SAC's entropy
  bonus on Day 6) can fail completely — plain DQN never once reaches the flag.
- **RND** turns "novelty" into an intrinsic reward that *directs* the agent to
  unexplored states, and it discovers the goal. In the lecture's taxonomy this
  **curiosity** family is *genuinely new* to deep RL — a cousin of bandit
  **optimism** (whose direct lift is the count-based / pseudo-count bonus): both
  reward what the agent is uncertain about.
- Two implementation details are essential and easy to forget: **input normalization**
  for the RND nets, and **running-std normalization** of the intrinsic reward.
- Same idea, other flavors from the lecture: pseudo-counts (density model) and ICM
  (forward/inverse dynamics). RND is simply the cheapest thing that works.